Data
├ Water Data
│ ├ data_2012.csv
│   ├ ymdhm : 년월일시분
│   ├ swl : 팔당댐 현재수위 (단위: El.m)
│   ├ inf : 팔당댐 유입량 (단위: m^3/s)
│   ├ sfw : 팔당댐 저수량 (단위: 만m^3)
│   ├ ecpc : 팔당댐 공용량 (단위: 백만m^3)
│   ├ tototf : 총 방류량 (단위: m^3/s)
│   ├ tide_level : 강화대교 조위 (단위: cm)
│   ├ wl_1018662 : 청담대교 수위 (단위: cm)
│   ├ fw_1018662 : 청담대교 유량 (단위: m^3/s)
│   ├ wl_1018680 : 잠수교 수위 (단위: cm)
│   ├ fw_1018680 : 잠수교 유량 (단위: m^3/s)
│   ├ wl_1018683 : 한강대교 수위 (단위: cm)
│   ├ fw_1018683 : 한강대교 유량 (단위: m^3/s)
│   ├ wl_1019630 : 행주대교 수위 (단위: cm)
│   └ fw_1019630 : 행주대교 유량 (단위: m^3/s)
│ ├ data_2013.csv
…
└ └ data_2022.csv
└ RainFall Data
│ ├ rf_2012.csv
│   ├ YMDHM : 년월일시분
│   ├ rf_10184100 : 대곡교 강수량
│   ├ rf_10184110 : 진관교 강수량
│   └ rf_10184140 : 송정동 강수량
│ ├ rf_2013.csv
…
└ └ rf_2022.csv

In [1]:
_input_path = '../data'
# _input_path = '/data'

import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt

def load_data(data_type):
    dataset_path = f'{_input_path}/{data_type}_data'
    dfs = []
    for file_name in sorted(os.listdir(f'{dataset_path}/')):
        if not file_name.endswith('.csv'):
            continue
        df = pd.read_csv(f'{dataset_path}/{file_name}', parse_dates=['ymdhm'])
        dfs.append( df )
    return pd.concat(dfs, axis=0, ignore_index=True)


def submit(Y_pred):
    '''
    Renders .csv file for submission.
    '''
    df = pd.read_csv(f'{_input_path}/sample_submission.csv')
    df['wl_1018662'] = Y_pred[:, 0]
    df['wl_1018680'] = Y_pred[:, 1]
    df['wl_1018683'] = Y_pred[:, 2]
    df['wl_1019630'] = Y_pred[:, 3]
    df.to_csv(f'../output/submission.csv', index=False)
    
def competition_metric(Y_true, Y_pred):
    '''
    Returns score for the current competition.
    '''
    l2_mean = np.mean(np.square(Y_true - Y_pred), axis=0)
    r_sq = 1 - (l2_mean / np.var(Y_true, axis=0))
    if np.isnan(r_sq).any() or (r_sq <= 0).any():
        return 999.
    return np.mean(np.sqrt(l2_mean) / r_sq)

In [2]:
def load_extended_data():
    wlobscds = ['1018662', '1018680', '1018683', '1019630']
    rfobscds = ['10184100', '10184110', '10184140']
    wlobscds += ['1018640', '1018645', '1018655', '1018658', '1018660', 
                '1018664', '1018669', '1018670', '1018675', '1018685',
                '1018695', '1018697']
    rfobscds += ['10184070', '10184080', '10184190', '10184200', '10194030']
    obs_post_ids = ['DT_0032']

    date = 20220822
    df_dm = pd.read_csv(
        f'{_input_path}/OpenAPI_merged/dm_{date}.csv', 
        parse_dates=['ymdhm'], 
        na_values=[' ', '##########'])

    df_wl = []
    for wlobscd in wlobscds:
        df_wl.append( pd.read_csv(
            f'{_input_path}/OpenAPI_merged/wl_{wlobscd}_{date}.csv', 
            na_values=[' ', '##########'])
        )
    df_wl = pd.concat(df_wl, axis=1).drop(columns=['ymdhm'])

    df_rf = []
    for rfobscd in rfobscds:
        df_rf.append( pd.read_csv(
            f'{_input_path}/OpenAPI_merged/rf_{rfobscd}_{date}.csv', 
            na_values=[' ', '##########']) 
                    )
    df_rf = pd.concat(df_rf, axis=1).drop(columns=['ymdhm'])

    df_tide = []
    for obs_post_id in obs_post_ids:
        df = pd.read_csv(f'{_input_path}/OpenAPI_merged/tide_level_{obs_post_id}_{date}.csv', parse_dates=['ymdhm'])
        df = df.drop_duplicates()
        df = df.set_index('ymdhm', drop=True)
        df = df.reindex(df.index.ceil('10min').drop_duplicates(), method='ffill')
        df = df.reindex(pd.date_range(pd.Timestamp(2012, 1, 1), pd.Timestamp(2022, 7, 31, 23, 50), freq='10min'))
        df = df.reset_index(drop=True)
        df_tide.append(df)
    df_tide = pd.concat(df_tide, axis=1)
    
    return pd.concat([df_dm, df_wl, df_tide, df_rf], axis=1)

In [3]:
def make_eraser(time, ser, bounds):
    eraser = (time < time.min())
    for (t0, t1, v0, v1) in bounds:
        if t0 is None: t0 = pd.Timestamp(2012, 5, 1)
        if t1 is None: t1 = pd.Timestamp(2022, 11, 1)
        if v0 is None: v0 = -np.inf
        if v1 is None: v1 = np.inf
        eraser[(time >= t0) & (time < t1) & (ser >= v0) & (ser < v1)] = True
    return eraser

_eraser_params = {
    'swl': [(None, None, None, 5)], 
    'inf': [(None, None, 20000, None), 
            (pd.Timestamp(2016, 5, 1), pd.Timestamp(2016, 6, 1), 17500, None), 
            (pd.Timestamp(2018, 10, 1), pd.Timestamp(2018, 11, 1), 3000, None)], 
    'sfw': [(None, None, None, 50)], 
    'ecpc': [(None, None, 200, None)],
    'tototf': [(None, None, 20000, None), 
               (pd.Timestamp(2016, 5, 1), pd.Timestamp(2016, 6, 1), 17500, None)],
#     'wl_1018683': [(pd.Timestamp(2022, 5, 1), None, None, 50)]
}

def rectify_sfw_ecpc(time, swl, sfw, ecpc):
    swl, sfw, ecpc = swl.copy(), sfw.copy(), ecpc.copy()
    mask_before_plunge = (time < pd.Timestamp(2015, 9, 1, 9, 30)) | (time == pd.Timestamp(2015, 9, 1, 10, 20)) \
        | ((time > pd.Timestamp(2015, 9, 1, 11, 10)) & (time < pd.Timestamp(2015, 9, 1, 12))) \
        | ((time > pd.Timestamp(2015, 9, 1, 12)) & (time < pd.Timestamp(2015, 9, 1, 13))) \
        | ((time > pd.Timestamp(2015, 9, 1, 13)) & (time < pd.Timestamp(2015, 9, 1, 13, 50)))
    A = np.stack([np.ones_like(swl), swl, mask_before_plunge.astype(float)], axis=1)
    B = np.stack([sfw, ecpc], axis=1)
    mask_not_na = ~(np.isnan(A).any(axis=1) | np.isnan(B).any(axis=1))
    drops = np.linalg.lstsq(A[mask_not_na], B[mask_not_na], rcond=None)[0][-1]
    sfw[mask_before_plunge] -= drops[0]
    ecpc[mask_before_plunge] -= drops[1]
    return sfw, ecpc

_tide_period = 12 + 25/60
_lunar_period = 29.53*24/2

def interpolate_tide(time, tide, from_, to_):
    n_pts_per_yr = 184*24*6
    n_pts_to_use = 1080
    min_pts_to_use = 540
    
    left_ = max(from_ - (from_ % n_pts_per_yr), from_ - n_pts_to_use)
    right_ = min(-(-to_ - ((-to_) % n_pts_per_yr)), to_ + n_pts_to_use)
    t = np.arange(left_, right_)
    daily_cpnts = [
        np.sin(2*np.pi*t/(_tide_period/2*6)), np.cos(2*np.pi*t/(_tide_period/2*6)), 
        np.sin(2*np.pi*t/(_tide_period*6)), np.cos(2*np.pi*t/(_tide_period*6)),
        np.sin(2*np.pi*t/(_tide_period*2*6)), np.cos(2*np.pi*t/(_tide_period*2*6)),
    ]
    monthly_cpnts = [
        np.sin(2*np.pi*t/(_lunar_period*6)), np.cos(2*np.pi*t/(_lunar_period*6)),
        np.sin(2*np.pi*t/(_lunar_period/2*6)), np.cos(2*np.pi*t/(_lunar_period/2*6)),
    ]
    mixed_cpnts = [
        d*m for d in daily_cpnts for m in monthly_cpnts
    ]
    A = np.stack([np.ones_like(t)] + daily_cpnts + monthly_cpnts + mixed_cpnts, axis=1)
    b = tide.iloc[left_:right_].values
    return (A @ np.linalg.lstsq(A[~np.isnan(b)], b[~np.isnan(b)], rcond=None)[0])[from_-left_:to_-left_]

def get_na_interval(ser, time=None):
    if time is None:
        time = ser.index
    begins = time[ser.isna().astype(int).diff() == 1]
    ends = time[ser.isna().astype(int).diff() == -1]
    return begins, ends

def interpolate_linearly(ser, width=None):
    if width is None:
        width = np.inf
    values = np.sort(ser[ser.notna()].unique())
    begins, ends = get_na_interval(ser)
    for bi, ei in zip(begins, ends):
        if ei - bi > width:
            continue
        bv, ev = ser.loc[bi-1], ser.loc[ei]
        mvs = np.linspace(bv, ev, ei-bi+2)[1:-1]
        idx_l = np.searchsorted(values, mvs)
        idx_r = np.minimum(idx_l + 1, len(values)-1)
        l_is_closer = ((mvs - values[idx_l]) <= (values[idx_r] - mvs))
        mvs[l_is_closer] = values[idx_l[l_is_closer]]
        mvs[~l_is_closer] = values[idx_r[~l_is_closer]]
        ser.loc[bi:ei-1] = mvs
    return ser
        
def cleanup(df):
    time = df['ymdhm']
    
#     tide_na_begins = df.index[df['tide_level'].isna().astype(int).diff() == 1]
#     tide_na_ends = df.index[df['tide_level'].isna().astype(int).diff() == -1]
#     for bi, ei in zip(tide_na_begins, tide_na_ends):
#         df.loc[bi:ei-1, 'tide_level'] = interpolate_tide(time, df['tide_level'], bi, ei)
    
    for col_name, params in _eraser_params.items():
        eraser = make_eraser(time, df[col_name], params)
        df.loc[eraser, col_name] = np.nan
    
    mask_unusual_spikes = (time >= pd.Timestamp(2017, 6, 20, 7)) & (time <= pd.Timestamp(2017, 6, 27, 17)) & (time.dt.minute == 0)
    mask_unusual_plateau = (time >= pd.Timestamp(2021, 10, 17)) & (time <= pd.Timestamp(2021, 10, 19, 15))
    df.loc[mask_unusual_spikes, ['swl', 'inf', 'sfw', 'ecpc', 'tototf']] = np.nan
#     for col_name in df.columns:
#         df[col_name] = interpolate_linearly(df[col_name].copy(), 1)
#     df.loc[mask_unusual_plateau, ['swl', 'inf', 'sfw', 'ecpc', 'tototf']] = np.nan
    
#     sfw, ecpc = rectify_sfw_ecpc(time, df['swl'], df['sfw'], df['ecpc'])
#     df['sfw'] = sfw
#     df['ecpc'] = ecpc
    return df

In [4]:
def features_for_learning(df):
    usable_features = [
        'swl', 'inf', 'sfw', 'ecpc',
        'tototf', 'tide_level_DT_0032', 
        'fw_1018662', 'fw_1018683', 'fw_1019630',
        'rf_10184100', 'rf_10184110', 'rf_10184140'
    ]
    removed_features = ['ymdhm', 'fw_1018680'] + [name for name in usable_features if name.startswith('wl')]
    usable_features = [name for name in df.columns if name not in removed_features]
    time = df['ymdhm']
    result = dict()
    for name in usable_features:
#         for i in range(1, 12+1):
        for i in range(1, 6+1):
            result[f'{name}_s{i}'] = df[name].shift(i)
            result[f'{name}_diff1_s{i}'] = df[name].diff(1).shift(i)
            result[f'{name}_diff2_s{i}'] = df[name].diff(2).shift(i)
#         for i in [3, 6, 9, 12, 15, 18]:
        for i in [6, 12]:
            result[f'{name}_roll{i}_mean'] = df[name].rolling(i).agg('mean').shift(1)
            result[f'{name}_roll{i}_median'] = df[name].rolling(i).agg('median').shift(1)
            result[f'{name}_roll{i}_std'] = df[name].rolling(i).agg('std').shift(1)
#             result[f'{name}_roll{i}_max'] = df[name].rolling(i).agg('max').shift(1)
#             result[f'{name}_roll{i}_min'] = df[name].rolling(i).agg('min').shift(1)
#             result[f'{name}_absdiff_roll{i}_mean'] = np.abs(df[name].diff()).rolling(i).agg('mean').shift(1)
#             result[f'{name}_diff_roll{i}_sqmean'] = np.square(df[name].diff()).rolling(i).agg('mean').shift(1)
#             result[f'{name}_absdiff_roll{i}_max'] = np.abs(df[name].diff()).rolling(i).agg('max').shift(1)
#             result[f'{name}_absdiff_roll{i}_min'] = np.abs(df[name].diff()).rolling(i).agg('min').shift(1)
            
    result['yr'] = time.dt.year
#     result['month'] = time.dt.month
#     result['dow'] = time.dt.day_of_week
    
    elapsed_index = (time - time.iloc[0]).dt.total_seconds() // 600
    pi = np.pi
    result['tide_period_cos'] = np.cos((2*pi/_tide_period)*elapsed_index)
    result['tide_period_sin'] = np.sin((2*pi/_tide_period)*elapsed_index)
#     result['tide_2period_cos'] = np.cos((pi/_tide_period)*elapsed_index)
#     result['tide_2period_sin'] = np.sin((pi/_tide_period)*elapsed_index)
    result['lunar_period_cos'] = np.cos((2*pi/_lunar_period)*elapsed_index)
    result['lunar_period_sin'] = np.sin((2*pi/_lunar_period)*elapsed_index)
    # temperature humidity etc
    # how to take delta into account
#     for name in ['wl_1018662', 'wl_1018680', 'wl_1018683', 'wl_1019630']:
#         for i in range(1, 6+1):
#             result[f'{name}_s{i}'] = remove_wrong_shift(df[name].shift(i), i)
    
    return pd.DataFrame(result)

def labels_for_learning(df):
    return df[['wl_1018662', 'wl_1018680', 'wl_1018683', 'wl_1019630']]

In [5]:
df = load_extended_data()
df_old = pd.merge(load_data('water'), load_data('rf'), on='ymdhm')
df_old = df_old.set_index('ymdhm', drop=True)
df_old = df_old.reindex(pd.date_range(pd.Timestamp(2012, 1, 1), pd.Timestamp(2022, 7, 31, 23, 50), freq='10min'))
df_old = df_old.reset_index().rename(columns={'index': 'ymdhm'})
# df = cleanup(df)

In [7]:
df.isna().sum(axis=1).value_counts()

10    198252
12    183792
11     91130
14     58795
13     12087
15      9580
19      1590
16      1035
17       102
20        70
18        67
26        31
25        27
21         1
30         1
dtype: int64

In [378]:
df_features = features_for_learning(df)
df_labels = labels_for_learning(df_old)

mask_train = (df['ymdhm'] < pd.Timestamp(2021, 6, 1))
mask_test = (df['ymdhm'] >= pd.Timestamp(2021, 6, 1)) & (df['ymdhm'] < pd.Timestamp(2021, 7, 19))
mask_train_final = (df['ymdhm'] < pd.Timestamp(2022, 6, 1))
mask_test_final = (df['ymdhm'] >= pd.Timestamp(2022, 6, 1)) & (df['ymdhm'] < pd.Timestamp(2022, 7, 19))

X_train, Y_train = df_features[mask_train], df_labels[mask_train]
X_test, Y_test = df_features[mask_test], df_labels[mask_test]

X_train_final, Y_train_final = df_features[mask_train_final], df_labels[mask_train_final]
X_test_final, _ = df_features[mask_test_final], _

In [379]:
df_interms = labels_for_learning(df)
I_train, I_test = df_interms[mask_train], df_interms[mask_test]
I_train_final = df_interms[mask_train_final]

In [381]:
import lightgbm as lgbm

class MyLGBM():
    def __init__(self, params):
        self.params = params
        
    def fit(self, X, Y):
        self.boosters = []
        for _, y in Y.iteritems():
            booster = lgbm.train(
                self.params,
                lgbm.Dataset(X, y),
            )
            self.boosters.append( booster )
        return self
    
    def predict(self, X):
        return np.stack([booster.predict(X) for booster in self.boosters], axis=1)

class Adapter():
    def __init__(self):
        pass
    
    def fit(self, X, Y):
        self.xs, self.ys = [], []
        for i in range(X.shape[1]):
            x, y = X.iloc[:, i], Y.iloc[:, i]
            x, y = x[~x.duplicated()], y[~x.duplicated()]
            mask_notna = ~np.isnan(x) & ~np.isnan(y)
            x, y = x[mask_notna], y[mask_notna]
            self.xs.append( x.iloc[x.argsort()] )
            self.ys.append( y.iloc[y.argsort()] )
        return self
    
    def transform(self, X):
        result = np.empty_like(X)
        for i in range(X.shape[1]):
            result[:, i] = np.interp(X[:, i], self.xs[i], self.ys[i])
        return result
            
params = {
    'max_depth': -1,
    'num_leaves': 31,
    'num_iterations': 100,
    'learning_rate': 1e-1,
    'force_row_wise': True,
    'random_state': 42,
    'deterministic': True,
    
    'verbosity': -1,
}

model = MyLGBM(params)
model.fit(X_train, I_train)

adapter = Adapter().fit(I_train, Y_train)
I_recon, I_pred = model.predict(X_train), model.predict(X_test)
Y_recon, Y_pred = adapter.transform(I_recon), adapter.transform(I_pred)
print(competition_metric(Y_train, Y_recon))
print(competition_metric(Y_test, Y_pred))

1.102042324313843
0.9112971741564657


In [380]:
model.fit(X_train_final, I_train_final)
adapter = Adapter().fit(I_train_final, Y_train_final)
I_pred_final = model.predict(X_test_final)
Y_pred_final = adapter.transform(I_pred_final)
submit(Y_pred_final)

In [51]:
start_new_search = False
all_features = list(df_features.columns)
usable_names = [
        'swl', 'inf', 'sfw', 'ecpc',
        'tototf', 'tide_level', 
        'fw_1018662', 'fw_1018683', 'fw_1019630',
        'rf_10184100', 'rf_10184110', 'rf_10184140'
    ]
if start_new_search:
    mask_current = np.full_like(df_features.columns, False, dtype=bool)
    default_features = []
    for name in usable_names:
        for i in range(1, 6+1):
            default_features.append( f'{name}_s{i}' )
            default_features.append( f'{name}_diff1_s{i}' )
        for i in [6, 12]:
            default_features.append( f'{name}_roll{i}_mean' ) 
            default_features.append( f'{name}_roll{i}_median' )
            default_features.append( f'{name}_roll{i}_std' )
        default_features += ['yr', 'tide_period_cos', 'tide_period_sin', 'lunar_period_cos', 'lunar_period_sin']
    for name in default_features:
        mask_current[all_features.index(name)] = True

model = MyLGBM(params)
model.fit(X_train.loc[:, mask_current], Y_train)
best_score = competition_metric(Y_test, model.predict(X_test.loc[:, mask_current]))
print(f'start with {best_score:.4f}')
block_size = 2
for i in range(n_iter := 10000):
    best_col_names = None
    col_names = np.random.choice(all_features, block_size, replace=False)
    mask_tmp = mask_current.copy()
    for col_name in col_names:
        idx = all_features.index(col_name)
        mask_tmp[idx] = ~mask_tmp[idx]
    model.fit(X_train.loc[:, mask_tmp], Y_train)
    Y_pred = model.predict(X_test.loc[:, mask_tmp])
    score = competition_metric(Y_test, Y_pred)
    if score < best_score:
        best_col_names, best_score = col_names, score
    if best_col_names is None:
        print('skipped')
        continue
    for col_name in best_col_names:
        idx = all_features.index(col_name)
        mask_current[idx] = ~mask_current[idx]
    print(f'toggle {best_col_names}')
    print(f'current score {score:.4f}')
    print()

start with 1.9393
skipped
skipped
skipped
toggle ['rf_10184110_diff2_s12' 'ecpc_diff_roll12_sqmean']
current score 1.9584

skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipped
skipp

KeyboardInterrupt: 

In [47]:
print(np.array(all_features)[mask_current])

['swl_s1' 'swl_diff1_s1' 'swl_s2' 'swl_diff1_s2' 'swl_s3' 'swl_diff1_s3'
 'swl_s4' 'swl_diff1_s4' 'swl_s5' 'swl_diff1_s5' 'swl_s6' 'swl_diff1_s6'
 'swl_diff1_s12' 'swl_roll6_mean' 'swl_roll6_median' 'swl_roll6_std'
 'swl_roll12_mean' 'swl_roll12_median' 'swl_roll12_std' 'inf_s1'
 'inf_diff1_s1' 'inf_s2' 'inf_diff1_s2' 'inf_s3' 'inf_diff1_s3' 'inf_s4'
 'inf_diff1_s4' 'inf_s5' 'inf_diff1_s5' 'inf_s6' 'inf_diff1_s6'
 'inf_roll6_mean' 'inf_roll6_median' 'inf_roll6_std' 'inf_roll12_mean'
 'inf_roll12_median' 'inf_roll12_std' 'sfw_s1' 'sfw_diff1_s1' 'sfw_s2'
 'sfw_diff1_s2' 'sfw_diff2_s2' 'sfw_s3' 'sfw_diff1_s3' 'sfw_diff2_s3'
 'sfw_s4' 'sfw_diff1_s4' 'sfw_s5' 'sfw_diff1_s5' 'sfw_s6' 'sfw_diff1_s6'
 'sfw_roll6_mean' 'sfw_roll6_median' 'sfw_roll6_std' 'sfw_roll9_mean'
 'sfw_roll12_mean' 'sfw_roll12_median' 'sfw_absdiff_roll12_min' 'ecpc_s1'
 'ecpc_diff1_s1' 'ecpc_s2' 'ecpc_diff1_s2' 'ecpc_s3' 'ecpc_diff1_s3'
 'ecpc_diff1_s4' 'ecpc_s5' 'ecpc_diff1_s5' 'ecpc_s6' 'ecpc_diff1_s6'
 'ecpc_diff_roll

['swl_s1' 'swl_diff1_s1' 'swl_s2' 'swl_diff1_s2' 'swl_s3' 'swl_diff1_s3'
 'swl_s4' 'swl_diff1_s4' 'swl_s5' 'swl_diff1_s5' 'swl_s6' 'swl_diff1_s6'
 'swl_diff1_s12' 'swl_roll6_mean' 'swl_roll6_median' 'swl_roll6_std'
 'swl_roll12_mean' 'swl_roll12_median' 'swl_roll12_std' 'inf_s1'
 'inf_diff1_s1' 'inf_s2' 'inf_diff1_s2' 'inf_s3' 'inf_diff1_s3' 'inf_s4'
 'inf_diff1_s4' 'inf_s5' 'inf_diff1_s5' 'inf_s6' 'inf_diff1_s6'
 'inf_roll6_mean' 'inf_roll6_median' 'inf_roll6_std' 'inf_roll12_mean'
 'inf_roll12_median' 'inf_roll12_std' 'sfw_s1' 'sfw_diff1_s1' 'sfw_s2'
 'sfw_diff1_s2' 'sfw_diff2_s2' 'sfw_s3' 'sfw_diff1_s3' 'sfw_diff2_s3'
 'sfw_s4' 'sfw_diff1_s4' 'sfw_s5' 'sfw_diff1_s5' 'sfw_s6' 'sfw_diff1_s6'
 'sfw_roll6_mean' 'sfw_roll6_median' 'sfw_roll6_std' 'sfw_roll9_mean'
 'sfw_roll12_mean' 'sfw_roll12_median' 'sfw_absdiff_roll12_min' 'ecpc_s1'
 'ecpc_diff1_s1' 'ecpc_s2' 'ecpc_diff1_s2' 'ecpc_s3' 'ecpc_diff1_s3'
 'ecpc_diff1_s4' 'ecpc_s5' 'ecpc_diff1_s5' 'ecpc_s6' 'ecpc_diff1_s6'
 'ecpc_diff_roll3_sqmean' 'ecpc_roll6_mean' 'ecpc_roll6_median'
 'ecpc_roll6_std' 'ecpc_roll12_mean' 'ecpc_roll12_median'
 'ecpc_roll12_std' 'ecpc_roll18_std' 'tototf_s1' 'tototf_diff1_s1'
 'tototf_s2' 'tototf_diff1_s2' 'tototf_s3' 'tototf_diff1_s3' 'tototf_s4'
 'tototf_diff1_s4' 'tototf_s5' 'tototf_diff1_s5' 'tototf_s6'
 'tototf_diff1_s6' 'tototf_roll6_mean' 'tototf_roll6_median'
 'tototf_roll6_std' 'tototf_roll12_mean' 'tototf_roll12_median'
 'tototf_roll12_std' 'tototf_absdiff_roll18_min' 'tide_level_s1'
 'tide_level_diff1_s1' 'tide_level_s2' 'tide_level_diff1_s2'
 'tide_level_s3' 'tide_level_diff1_s3' 'tide_level_diff1_s4'
 'tide_level_s5' 'tide_level_diff1_s5' 'tide_level_s6'
 'tide_level_diff1_s6' 'tide_level_roll6_mean' 'tide_level_roll6_median'
 'tide_level_roll6_std' 'tide_level_roll12_mean'
 'tide_level_roll12_median' 'tide_level_roll12_std' 'fw_1018662_s1'
 'fw_1018662_diff1_s1' 'fw_1018662_s2' 'fw_1018662_diff1_s2'
 'fw_1018662_s3' 'fw_1018662_diff1_s3' 'fw_1018662_s4'
 'fw_1018662_diff1_s4' 'fw_1018662_s5' 'fw_1018662_diff1_s5'
 'fw_1018662_s6' 'fw_1018662_diff1_s6' 'fw_1018662_roll6_mean'
 'fw_1018662_roll6_median' 'fw_1018662_roll6_std' 'fw_1018662_roll12_mean'
 'fw_1018662_roll12_median' 'fw_1018662_roll12_std' 'fw_1018683_s1'
 'fw_1018683_diff1_s1' 'fw_1018683_s2' 'fw_1018683_diff1_s2'
 'fw_1018683_s3' 'fw_1018683_diff1_s3' 'fw_1018683_s4'
 'fw_1018683_diff1_s4' 'fw_1018683_s5' 'fw_1018683_diff1_s5'
 'fw_1018683_s6' 'fw_1018683_diff1_s6' 'fw_1018683_roll6_mean'
 'fw_1018683_roll6_median' 'fw_1018683_roll6_std' 'fw_1018683_roll12_mean'
 'fw_1018683_roll12_median' 'fw_1018683_roll12_std' 'fw_1019630_s1'
 'fw_1019630_diff1_s1' 'fw_1019630_s2' 'fw_1019630_diff1_s2'
 'fw_1019630_s3' 'fw_1019630_diff1_s3' 'fw_1019630_s4'
 'fw_1019630_diff1_s4' 'fw_1019630_s5' 'fw_1019630_diff1_s5'
 'fw_1019630_s6' 'fw_1019630_diff1_s6' 'fw_1019630_roll6_mean'
 'fw_1019630_roll6_median' 'fw_1019630_roll6_std' 'fw_1019630_roll12_mean'
 'fw_1019630_roll12_median' 'fw_1019630_roll12_std' 'rf_10184100_s1'
 'rf_10184100_diff1_s1' 'rf_10184100_s2' 'rf_10184100_diff1_s2'
 'rf_10184100_s3' 'rf_10184100_diff1_s3' 'rf_10184100_s4'
 'rf_10184100_diff1_s4' 'rf_10184100_s5' 'rf_10184100_diff1_s5'
 'rf_10184100_s6' 'rf_10184100_diff1_s6' 'rf_10184100_roll6_mean'
 'rf_10184100_roll6_median' 'rf_10184100_roll6_std'
 'rf_10184100_roll12_mean' 'rf_10184100_roll12_median'
 'rf_10184100_roll12_std' 'rf_10184100_absdiff_roll15_max'
 'rf_10184110_s1' 'rf_10184110_diff1_s1' 'rf_10184110_s2'
 'rf_10184110_diff1_s2' 'rf_10184110_s3' 'rf_10184110_diff1_s3'
 'rf_10184110_s4' 'rf_10184110_diff1_s4' 'rf_10184110_s5'
 'rf_10184110_diff1_s5' 'rf_10184110_s6' 'rf_10184110_diff1_s6'
 'rf_10184110_roll6_mean' 'rf_10184110_roll6_median'
 'rf_10184110_roll6_std' 'rf_10184110_roll12_mean'
 'rf_10184110_roll12_median' 'rf_10184110_roll12_std'
 'rf_10184110_roll12_max' 'rf_10184140_s1' 'rf_10184140_diff1_s1'
 'rf_10184140_s2' 'rf_10184140_diff1_s2' 'rf_10184140_s3'
 'rf_10184140_diff1_s3' 'rf_10184140_s4' 'rf_10184140_diff1_s4'
 'rf_10184140_s5' 'rf_10184140_diff1_s5' 'rf_10184140_s6'
 'rf_10184140_diff1_s6' 'rf_10184140_roll6_mean'
 'rf_10184140_roll6_median' 'rf_10184140_roll6_std'
 'rf_10184140_roll12_mean' 'rf_10184140_roll12_median'
 'rf_10184140_roll12_std' 'yr' 'tide_period_cos' 'tide_period_sin'
 'lunar_period_cos' 'lunar_period_sin']

In [ ]:
# class MyLGBM():
#     def __init__(self, params):
#         self.params = params
        
#     def fit(self, X, Y):
#         self.boosters = []
#         self.boosters_lower = []
#         for _, y in Y.iteritems():
#             booster = lgbm.train(
#                 self.params,
#                 lgbm.Dataset(X, y)
#             )
#             self.boosters.append( booster )
#         X_tmp = self._intermediate(X)
#         for _, y in Y.iteritems():
#             booster = lgbm.train(
#                 self.params,
#                 lgbm.Dataset(X_tmp, y)
#             )
#             self.boosters_lower.append( booster )
#         return self
    
#     def predict(self, X):
#         X_tmp = self._intermediate(X)
#         Y = np.stack([booster.predict(X_tmp) for booster in self.boosters_lower], axis=1)
#         return Y
    
#     def _intermediate(self, X):
#         Y_tmp = np.stack([booster.predict(X) for booster in boosters], axis=1)
#         X_tmp = pd.concat([X] + [pd.DataFrame(Y_tmp, index=X.index, columns=[f'p{j}s{d}' for j in range(4)]).shift(d) for d in range(6)], axis=1)
#         return X_tmp